# Xarray-Spatial Edge Detection: Sobel, Prewitt, and Laplacian

Edge detection filters pick out boundaries and transitions in raster data. In remote sensing you might use them to find field boundaries, road edges, or geological contacts. In terrain analysis they highlight ridgelines and breaks in slope. This notebook covers the five filters in `xrspatial.edge_detection`, all of which are thin wrappers around the existing `convolution_2d` infrastructure.

### What you'll build

1. Generate synthetic terrain and a hillshade base layer
2. Run Sobel X and Sobel Y to extract directional gradients
3. Combine them into an edge magnitude map
4. Compare Sobel against Prewitt on the same terrain
5. Apply the Laplacian for omnidirectional second-derivative edges
6. Visualize boundary mode differences

![Edge detection preview](images/edge_detection_preview.png)

[Sobel X and Sobel Y](#Sobel-X-and-Sobel-Y) · [Edge magnitude](#Edge-magnitude) · [Sobel vs. Prewitt](#Sobel-vs.-Prewitt) · [Laplacian](#Laplacian) · [Boundary modes](#Boundary-modes)

Standard imports plus the edge detection functions.

In [ ]:
%matplotlib inline
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import xrspatial
from xrspatial.edge_detection import sobel_x, sobel_y, laplacian, prewitt_x, prewitt_y

## Terrain data

Synthetic elevation from `generate_terrain`, reused in every section below.

In [ ]:
W, H = 800, 600
x_range = (-20e6, 20e6)
y_range = (-20e6, 20e6)

terrain = xr.DataArray(np.zeros((H, W)))
terrain = terrain.xrs.generate_terrain(x_range=x_range, y_range=y_range)
hillshade = terrain.xrs.hillshade()

terrain.plot.imshow(cmap='terrain', size=7.5, aspect=W/H, add_colorbar=False)

Blues are valleys, greens and browns rise to ridges and peaks. The terrain has enough topographic variety to exercise all five edge filters.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
terrain.plot.imshow(ax=ax, cmap='terrain', alpha=128/255, add_colorbar=False)
ax.set_axis_off()

## Sobel X and Sobel Y

The [Sobel operator](https://en.wikipedia.org/wiki/Sobel_operator) approximates the image gradient using a pair of 3x3 kernels. `sobel_x` responds to horizontal changes (it lights up vertical edges), and `sobel_y` responds to vertical changes (horizontal edges). Both output signed values: positive on one side of the edge, negative on the other.

The two panels below show the X and Y gradients side by side on a diverging colormap.

In [ ]:
sx = sobel_x(terrain, boundary='nearest')
sy = sobel_y(terrain, boundary='nearest')

vmax = max(abs(float(sx.min())), abs(float(sx.max())),
           abs(float(sy.min())), abs(float(sy.max())))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, data, title in zip(axes, [sx, sy], ['Sobel X (horizontal gradient)', 'Sobel Y (vertical gradient)']):
    im = data.plot.imshow(ax=ax, cmap='RdBu_r', vmin=-vmax, vmax=vmax, add_colorbar=False)
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()

fig.colorbar(im, ax=axes, shrink=0.8, label='Gradient response')
plt.tight_layout()

Notice how Sobel X picks up the left/right faces of ridges while Sobel Y picks up the top/bottom faces. Flat areas are near zero (white). The sign tells you which side of the edge you're on.

## Edge magnitude

Neither Sobel X nor Sobel Y alone captures all edges. The standard approach is to combine them into a single magnitude: `sqrt(sobel_x**2 + sobel_y**2)`. This gives an unsigned edge strength that responds to transitions in any direction.

The plot overlays edge magnitude on the hillshade so you can see how the detected edges line up with terrain features.

In [ ]:
magnitude = np.sqrt(sx**2 + sy**2)

fig, ax = plt.subplots(figsize=(10, 7.5))
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
im = magnitude.plot.imshow(ax=ax, cmap='inferno', alpha=200/255, add_colorbar=False)
ax.set_title('Sobel edge magnitude', fontsize=14)
ax.set_axis_off()
fig.colorbar(im, ax=ax, shrink=0.7, label='Edge strength')

Bright pixels mark steep transitions. Ridgelines and valley walls light up, flat plains stay dark. This is equivalent to what `xrspatial.slope` computes (with different scaling), since slope is also a gradient magnitude.

## Sobel vs. Prewitt

Sobel and Prewitt both compute directional gradients with 3x3 kernels. The difference is how they weight the center row/column: Sobel uses `[1, 2, 1]` (more smoothing perpendicular to the gradient direction), Prewitt uses `[1, 1, 1]` (uniform weight). In practice the results are similar, with Sobel slightly less sensitive to diagonal noise.

The top row shows the X-gradient from each operator. The bottom row shows the difference and a cross-section to quantify it.

In [ ]:
px = prewitt_x(terrain, boundary='nearest')
py = prewitt_y(terrain, boundary='nearest')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top row: Sobel X vs Prewitt X
grad_max = max(abs(float(sx.min())), abs(float(sx.max())),
               abs(float(px.min())), abs(float(px.max())))

for ax, data, title in zip(axes[0], [sx, px], ['Sobel X', 'Prewitt X']):
    data.plot.imshow(ax=ax, cmap='RdBu_r', vmin=-grad_max, vmax=grad_max, add_colorbar=False)
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()

# Bottom left: difference map
diff = sx - px
diff_max = max(abs(float(diff.min())), abs(float(diff.max())))
im = diff.plot.imshow(ax=axes[1][0], cmap='RdBu_r', vmin=-diff_max, vmax=diff_max, add_colorbar=False)
axes[1][0].set_title('Sobel X \u2212 Prewitt X', fontsize=13)
axes[1][0].set_axis_off()
fig.colorbar(im, ax=axes[1][0], shrink=0.7, label='Difference')

# Bottom right: cross-section
row = H // 2
ax = axes[1][1]
cols = np.arange(W)
ax.plot(cols, sx.values[row], label='Sobel X', color='#2166ac', linewidth=1.2)
ax.plot(cols, px.values[row], label='Prewitt X', color='#b2182b', linewidth=1.2, linestyle='--')
ax.set_title(f'Cross-section at row {row}', fontsize=13)
ax.set_xlabel('Column')
ax.set_ylabel('Gradient response')
ax.legend(fontsize=10)

plt.tight_layout()

The two operators track each other closely. The difference is largest where the gradient changes rapidly (steep ridgelines), because that's where Sobel's extra center weight produces a slightly smoother estimate. For most applications either one works fine.

## Laplacian

The [Laplacian](https://en.wikipedia.org/wiki/Discrete_Laplace_operator) is a second-derivative operator. Instead of measuring the gradient direction like Sobel/Prewitt, it measures the rate of change of the gradient itself. It responds to edges in all directions at once and crosses zero right at the edge, which makes it useful for finding exact edge locations.

The trade-off: it responds to noise more aggressively than first-derivative operators because second derivatives amplify high-frequency content.

In [ ]:
lap = laplacian(terrain, boundary='nearest')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Laplacian response
lap_max = max(abs(float(lap.min())), abs(float(lap.max())))
im = lap.plot.imshow(ax=axes[0], cmap='RdBu_r', vmin=-lap_max, vmax=lap_max, add_colorbar=False)
axes[0].set_title('Laplacian', fontsize=13)
axes[0].set_axis_off()
fig.colorbar(im, ax=axes[0], shrink=0.7, label='Second derivative')

# Compare: Sobel magnitude vs abs(Laplacian)
ax = axes[1]
row = H // 2
cols = np.arange(W)
sobel_mag_row = magnitude.values[row]
lap_abs_row = np.abs(lap.values[row])

# Normalize both to [0, 1] for visual comparison
ax.plot(cols, sobel_mag_row / sobel_mag_row.max(), label='Sobel magnitude (normalized)',
        color='#2166ac', linewidth=1.2)
ax.plot(cols, lap_abs_row / lap_abs_row.max(), label='|Laplacian| (normalized)',
        color='#b2182b', linewidth=1.2)
ax.set_title(f'First vs. second derivative at row {row}', fontsize=13)
ax.set_xlabel('Column')
ax.set_ylabel('Normalized response')
ax.legend(fontsize=10)

plt.tight_layout()

The cross-section shows the two operators responding to the same features but with different shapes. The Sobel magnitude peaks at the edge, while the Laplacian has twin peaks that straddle the edge with a zero-crossing in between. That zero-crossing property is why the Laplacian is used in "Laplacian of Gaussian" edge detectors to locate edge positions with sub-pixel precision.

<div class="alert alert-block alert-warning">
<b>Noise sensitivity.</b> The Laplacian amplifies noise because it is a second derivative. If your raster has sensor noise or quantization artifacts, consider smoothing it first (e.g. with <code>xrspatial.focal.mean</code> or a Gaussian convolution) before applying the Laplacian.
</div>

## Boundary modes

All five filters accept a `boundary` parameter that controls how to handle pixels at the raster edge where the 3x3 kernel extends beyond the data. The default `'nan'` leaves a 1-pixel NaN border. The alternatives (`'nearest'`, `'reflect'`, `'wrap'`) fill in values so you get results all the way to the edge.

The four panels below show Sobel X with each boundary mode, zoomed into the top-left corner to make the difference visible.

In [ ]:
modes = ['nan', 'nearest', 'reflect', 'wrap']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
zoom = (slice(0, 80), slice(0, 120))  # top-left corner

for ax, mode in zip(axes, modes):
    result = sobel_x(terrain, boundary=mode)
    zoomed = result[zoom]
    zoomed.plot.imshow(ax=ax, cmap='RdBu_r', add_colorbar=False)
    ax.set_title(f"boundary='{mode}'", fontsize=12)
    ax.set_axis_off()

plt.suptitle('Sobel X, top-left corner zoom', fontsize=14, y=1.02)
plt.tight_layout()

<div class="alert alert-block alert-info">
<b>Which mode to use?</b> For terrain and imagery, <code>'nearest'</code> or <code>'reflect'</code> are good defaults since they avoid the NaN border without introducing discontinuities. <code>'wrap'</code> is mainly useful for periodic data (global grids that wrap around the antimeridian, for example).
</div>

In [ ]:
import pathlib

# Generate preview image: 4-panel showing all filter families
prewitt_mag = np.sqrt(px**2 + py**2)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

panels = [
    (sx, 'Sobel X', 'RdBu_r'),
    (sy, 'Sobel Y', 'RdBu_r'),
    (magnitude, 'Sobel magnitude', 'inferno'),
    (lap, 'Laplacian', 'RdBu_r'),
]

for ax, (data, title, cmap) in zip(axes, panels):
    if cmap == 'RdBu_r':
        vm = max(abs(float(data.min())), abs(float(data.max())))
        data.plot.imshow(ax=ax, cmap=cmap, vmin=-vm, vmax=vm, add_colorbar=False)
    else:
        data.plot.imshow(ax=ax, cmap=cmap, add_colorbar=False)
    ax.set_title(title, fontsize=11)
    ax.set_axis_off()

plt.tight_layout()
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/edge_detection_preview.png', bbox_inches='tight', dpi=120)
plt.close(fig)

### References

- [Sobel operator](https://en.wikipedia.org/wiki/Sobel_operator), Wikipedia
- [Prewitt operator](https://en.wikipedia.org/wiki/Prewitt_operator), Wikipedia
- [Discrete Laplace operator](https://en.wikipedia.org/wiki/Discrete_Laplace_operator), Wikipedia
- [Image gradient](https://en.wikipedia.org/wiki/Image_gradient), Wikipedia
- [Edge detection](https://en.wikipedia.org/wiki/Edge_detection), Wikipedia